<a href="https://colab.research.google.com/github/yuri-maradini/TempSal/blob/main/src/train_ueyes_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning TempSAL su UEyes — Colab

Notebook pronto per lanciare il training vero (Step 4) su GPU, invece che sulla CPU locale.

**Prima di eseguire questo notebook**, su Google Drive crea una cartella (default atteso: `MyDrive/TempSAL_UEyes/`) contenente:
- `multilevel_tempsal.pt` — il checkpoint pre-addestrato originale
- `data_ueyes.zip` — l'archivio di `data_ueyes/` generato in locale (Step 1-2)

Poi: **Runtime → Cambia tipo di runtime → GPU**, prima di eseguire le celle.

In [1]:
# Controllo che sia stata assegnata una GPU
!nvidia-smi

Sat Sep 12 12:12:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# Cambia questo path se hai usato un nome/percorso diverso su Drive
DRIVE_DIR = '/content/drive/MyDrive/TempSAL_UEyes'

assert os.path.isdir(DRIVE_DIR), (
    f"Cartella non trovata: {DRIVE_DIR}\n"
    "Creala su Drive e caricaci multilevel_tempsal.pt + data_ueyes.zip prima di continuare."
)
print('Contenuto trovato su Drive:', os.listdir(DRIVE_DIR))

Contenuto trovato su Drive: ['multilevel_tempsal.pt', 'data_ueyes.zip', 'multilevel_tempsal_ueyes.pt', 'multilevel_tempsal_ueyes_v2.pt', 'multilevel_tempsal_ueyes_v3.pt', 'multilevel_tempsal_ueyes_v4.pt']


In [4]:
# Codice: sempre aggiornato da GitHub, non serve preparazione
!git clone https://github.com/yuri-maradini/TempSal.git /content/TempSal

Cloning into '/content/TempSal'...
remote: Enumerating objects: 264, done.
remote: Counting objects: 100% (264/264), done.
remote: Compressing objects: 100% (202/202), done.
remote: Total 264 (delta 104), reused 215 (delta 58), pack-reused 0 (from 0)
Receiving objects: 100% (264/264), 10.32 MiB | 25.11 MiB/s, done.
Resolving deltas: 100% (104/104), done.


In [5]:
!cd /content/TempSal && git pull


Already up to date.


In [6]:
import shutil

os.makedirs('/content/TempSal/src/checkpoints', exist_ok=True)
shutil.copy(
    f'{DRIVE_DIR}/multilevel_tempsal.pt',
    '/content/TempSal/src/checkpoints/multilevel_tempsal.pt',
)
# multilevel_tempsal_ueyes_v4.pt e' il checkpoint della quarta run (backbone
# sbloccato, statistiche BatchNorm congelate, il migliore sul ramo temporale
# finora): questa run riparte da li'.
shutil.copy(
    f'{DRIVE_DIR}/multilevel_tempsal_ueyes_v4.pt',
    '/content/TempSal/src/checkpoints/multilevel_tempsal_ueyes_v4.pt',
)
print('Checkpoint copiati (originale + v4, il warm-start di questa run).')

Checkpoint copiati (originale + v4, il warm-start di questa run).


In [7]:
import time
import zipfile

# Estratto sul disco locale di Colab (veloce), non lasciato sul mount di Drive
# (l'I/O su Drive montato e' molto piu' lento per tanti file piccoli, e qui
# ce ne sono migliaia tra immagini, mappe e volumi temporali).
t0 = time.time()
with zipfile.ZipFile(f'{DRIVE_DIR}/data_ueyes.zip') as zf:
    zf.extractall('/content/TempSal/')
print(f'Dati estratti in {time.time() - t0:.0f}s')

Dati estratti in 25s


In [8]:
# Controllo veloce di integrita': i conteggi devono combaciare con quelli
# verificati in locale (1872 train / 108 val per ciascuna sottocartella)
for sub in ['images', 'maps', 'fixation_maps', 'saliency_volumes_5', 'fixation_volumes_5']:
    for split in ['train', 'val']:
        d = f'/content/TempSal/data_ueyes/{sub}/{split}'
        n = len(os.listdir(d)) if os.path.isdir(d) else 'MANCANTE'
        print(f'{sub:22s} {split:5s} -> {n}')

images                 train -> 1872
images                 val   -> 108
maps                   train -> 1872
maps                   val   -> 108
fixation_maps          train -> 1872
fixation_maps          val   -> 108
saliency_volumes_5     train -> 9360
saliency_volumes_5     val   -> 540
fixation_volumes_5     train -> 9360
fixation_volumes_5     val   -> 540


In [9]:
# Colab ha gia' PyTorch con supporto CUDA preinstallato: installiamo solo le
# altre dipendenze del progetto, senza toccare torch/torchvision/torchaudio
# (forzare i pin usati in locale, pensati per una build CPU, rischierebbe di
# rimpiazzare la build CUDA gia' pronta di Colab con una incompatibile).
!pip install -q wandb pycocotools ftfy einops clip-anytorch kornia regex

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 98.5 MB/s eta 0:00:00


In [10]:
# Verifica che l'installazione sopra non abbia rovinato il supporto CUDA di torch
import torch
print('torch', torch.__version__, '| CUDA disponibile:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU non disponibile: controlla Runtime > Cambia tipo di runtime > GPU'

torch 2.11.0+cu128 | CUDA disponibile: True


## wandb (consigliato per questa run)

Il primo run (10 epoche) è stato fatto con `WANDB_MODE=disabled`: nessuna curva salvata, i numeri per epoca sono stati recuperati a mano dall'output della cella di training. Per questa run vale la pena accendere il logging vero, così le curve di CC/KLDIV/NSS/SIM (aggregate e per-slice) restano disponibili per il confronto e per la tesi senza dover rileggere l'output della cella.

Esegui la cella sotto (chiede l'API key, la trovi su wandb.ai/authorize) prima di lanciare il training. Se preferisci comunque saltarlo, aggiungi di nuovo `WANDB_MODE=disabled` (o `=offline` per salvare i log in locale senza account) davanti al comando `python train.py` nella cella di training.</cell id="HNcrR_LSgcXM">


In [11]:
import wandb
wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yurimaradini (yurimaradini-universit-di-padova) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Training (run 5 — più epoche + learning rate differenziato sul mixing)

Le run precedenti hanno stabilito: la mappa aggregata satura presto (~epoca 10 della run 1/2), il ramo temporale continua a migliorare più a lungo, e sbloccare il backbone (v4) aiuta il ramo temporale ma lascia la mappa aggregata leggermente **sotto** v2 — probabile causa (mai testata finora): il decoder di mixing (`deconv_layer1..4` + `deconv_mix`, sempre allenabile) riceve come input anche le slice temporali prodotte da `pnas_vol`, che in v4 sono cambiate sotto di lui; il decoder si allena però alla stessa velocità lentissima (1e-6) del backbone appena sbloccato, e non ha ancora fatto in tempo a recalibrarsi.

**Questa run testa l'ipotesi direttamente**, invece di un'altra run "a scatola chiusa":
- Nuovo argomento in `train.py`, `--mixing_lr`: dà al decoder di mixing un learning rate proprio, separato da quello di backbone/testa temporale. Se non passato, ricade su `--lr` (comportamento identico alle run precedenti — nessuna run passata è invalidata da questa modifica).
- `--mixing_lr 1e-5`: non è un valore arbitrario — è lo stesso learning rate con cui il decoder di mixing si è già allenato con successo nelle run 1 e 2 (100 volte più alto di `--lr 1e-6` usato per backbone/testa in v3/v4), quindi è un valore già verificato sicuro per questi stessi layer.
- `--no_epochs 20` (raddoppiato rispetto a v4): il punteggio non aveva ancora raggiunto un plateau netto all'epoca 9 di v4, quindi c'è margine residuo sul ramo temporale, e il decoder di mixing ha bisogno di più tempo per recalibrarsi.
- Warm-start da `multilevel_tempsal_ueyes_v4.pt` (il miglior checkpoint sul ramo temporale finora), non da v2 — si continua dal punto più avanzato, non si riparte da zero.
- Tutto il resto invariato da v3/v4: `--train_enc 1`, `--train_model 1`, `--lr 1e-6` per backbone e testa, `--batch_size 16 --grad_accum_steps 2` (stesso fix OOM), `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`.

**Perché in un'unica run e non in due run separate** (prima più epoche, poi il mixing_lr): cambiare due variabili in due run distinte renderebbe impossibile isolare quale delle due abbia causato un eventuale miglioramento — esattamente il tipo di confusione che il metodo seguito finora (una variabile alla volta) ha sempre evitato. Inoltre non c'è un vero bisogno di aspettare: il decoder di mixing può rincorrere le slice temporali via gradiente man mano che cambiano, non serve che il ramo temporale abbia "finito" prima.

**Verificato in locale su CPU** prima di lanciare su Colab: con `--mixing_lr` esplicito, l'ottimizzatore crea correttamente due gruppi di parametri (`793 tensori a lr=1e-06, 14 tensori (mixing decoder) a lr=1e-05`) — il conteggio di 14 tensori nel gruppo mixing coincide esattamente con la verifica peso-per-peso già fatta nello Step 3 (`deconv_layer1..4` + `deconv_mix` = 14 tensori allenabili). Verificata anche la retrocompatibilità: senza passare `--mixing_lr`, i due gruppi ricadono sullo stesso learning rate e il comportamento è identico a prima.

**Da controllare dopo la run**: se la mappa aggregata torna al livello di v2 (o lo supera) — confermerebbe l'ipotesi del decoder di mixing "in ritardo"; se resta comunque piatta, l'ipotesi va scartata e la spiegazione della slide S14 andrà rivista. Da monitorare anche il gap train/val con più attenzione: in v4 la loss di training scendeva mentre la mappa aggregata era già piatta, un primo segnale di overfitting silenzioso — con 20 epoche invece di 10 il rischio è più concreto.

In [12]:
%cd /content/TempSal/src
# PYTORCH_CUDA_ALLOC_CONF: riduce la frammentazione dell'allocatore, utile
# ora che la memoria e' molto piu' vicina al limite della T4 (vedi cella
# markdown sulla run 4 sull'OOM del primo tentativo).
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train.py \
  --enc_model pnas_boosted_multi \
  --dataset_dir ../data_ueyes/ \
  --model_path ./checkpoints/multilevel_tempsal_ueyes_v4.pt \
  --model_vol_path ./checkpoints/multilevel_tempsal_ueyes_v4.pt \
  --train_model 1 \
  --train_enc 1 \
  --lr 1e-6 \
  --mixing_lr 1e-5 \
  --batch_size 16 \
  --grad_accum_steps 2 \
  --no_epochs 20 \
  --model_val_path ./checkpoints/multilevel_tempsal_ueyes_v5.pt

/content/TempSal/src
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: yurimaradini (yurimaradini-universit-di-padova) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/TempSal/src/wandb/run-20260912_121441-c25afgof
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run likely-wood-7
wandb: ⭐️ View project at https://wandb.ai/yurimaradini-universit-di-padova/tempsal-ueyes
wandb: 🚀 View run at https://wandb.ai/yurimaradini-universit-di-padova/tempsal-ueyes/runs/c25afgof
PNAS Boosted Model PNASBoostedModelMultiLevel
96 96 54
96 270 108
270 540 216
540 1080 216
1080 1080 216
1080 1080 216
1080 1

In [13]:
# Copia il checkpoint fine-tuned su Drive, cosi' sopravvive alla chiusura
# della sessione Colab. Puoi rieseguire questa cella anche a training ancora
# in corso, per avere un backup intermedio.
import shutil

src_ckpt = '/content/TempSal/src/checkpoints/multilevel_tempsal_ueyes_v5.pt'
dst_ckpt = f'{DRIVE_DIR}/multilevel_tempsal_ueyes_v5.pt'
shutil.copy(src_ckpt, dst_ckpt)
print('Copiato su Drive:', dst_ckpt)

Copiato su Drive: /content/drive/MyDrive/TempSAL_UEyes/multilevel_tempsal_ueyes_v5.pt
